<a href="https://colab.research.google.com/github/eliabrodsky/la_data/blob/main/Hospital_Analysis_VBC_EB_Sept_2026_rev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Louisiana Rural Hospital Sustainability and VBC Readiness

Segmenting Louisiana's rural hospitals on financial position, and assessing which
are positioned to enter value-based payment arrangements.

Data: CMS Hospital Provider Cost Reports (HCRIS, form CMS-2552-10), FY2021–FY2023.

---
# 1 &nbsp; Logic of the analysis

## The question

Which Louisiana rural hospitals can sustain themselves, what distinguishes them
from those that cannot, and which are positioned to take on value-based care.

## The reasoning the analysis is built on

A rural hospital's financial position is driven by three things, in roughly this order:

1. **The reimbursement class it holds.** A Critical Access Hospital is paid close to
   cost. An IPPS hospital gets a fixed price per case. That difference dominates the
   income statement before any management decision is made.
2. **The ownership structure it sits inside.** A parish hospital district holds its
   own cash. A hospital inside a corporate system has its cash swept to the parent.
   The balance sheet reflects the corporate arrangement, not the hospital's health.
3. **Its actual operating performance.** What is left once the first two are
   accounted for.

Value-based care participation requires a **fourth** thing that is unrelated to all
three: control of primary care billing. Patients are attributed to whoever bills
their primary care visits, so a hospital with no employed primary care providers
brings no attributable lives regardless of how large or how solvent it is.

The analysis exists to separate these, so that a hospital is not mistaken for a
strong operator when it is really a well-reimbursed one, and not mistaken for a VBC
candidate when it simply has beds.

## Why each step is in the notebook

| Step | Purpose |
|---|---|
| **2 Load** | Build a three-year panel and derive comparable ratios |
| **3 Explore** | Establish which metrics are comparable across hospitals and which are contaminated by structure |
| **4 Cluster** | Find groups that cut across the categories we already have, and test whether those groups are real |
| **5 Interpret** | Name what separates the groups, and state what the analysis does not answer |

## Two design decisions worth stating up front

**Trajectory, not snapshot.** Segments assigned from a single year hold across three
years for only about half of these hospitals. Cost-settled hospitals swing on
settlement timing, so a one-year label for a named hospital is close to a coin flip.
Features are therefore built as level, slope and volatility across all three years.

**Stratify, do not cluster, on structure.** Reimbursement class and ownership are
fed in as context, never as clustering inputs. If they go into the model, the
algorithm simply recovers categories we already knew, and the exercise tells us
nothing new.

## Definitions

| Term | Meaning here |
|---|---|
| **Rural hospital** | The 64-hospital universe with a Medicare cost report, drawn from the CMS CAH list and the LDH rural designation lists |
| **Reimbursement class** | Medicare class: CAH, IPPS, SCH, RRC, REH |
| **Ownership stratum** | Type of Control from Worksheet S-2, collapsed to Government, Nonprofit, Proprietary |
| **Sustainability** | Ability to fund operations and reinvestment from recurring revenue, whether that revenue is patient care or policy |
| **VBC potential** | Risk-bearing capacity **and** operating performance **and** attributable primary care lives. All three required, none sufficient |

---
# 2 &nbsp; Loading the data

## 2.1 &nbsp; Environment

Standard scientific stack plus scikit-learn. Plot defaults are set once here so
every figure downstream renders consistently.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import kruskal, linregress
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import cdist

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold

warnings.filterwarnings('ignore', category=FutureWarning)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('Environment ready.')

## 2.2 &nbsp; Source files

Three annual CMS cost report public use files, read directly from GitHub so the
notebook runs from a clean Colab runtime with nothing to upload.

`rural_ccn.csv` is the crosswalk that defines the 64-hospital rural universe and
carries the LIHNC and Epic pipeline flags. It also resolves filing-name mismatches,
since hospitals file under legal names that differ from how they are known:
Prevost is West Ascension, Richland Parish Hospital Service District is Richardson,
Riverland is Trinity.

In [ ]:
BASE = 'https://raw.githubusercontent.com/eliabrodsky/la_data/main/'

FILES = {
    2021: BASE + 'CostReport_2021_Final.csv',
    2022: BASE + 'CostReport_2022_Final.csv',
    2023: BASE + 'CostReport_2023_Final.csv',
}

CROSSWALK = BASE + 'rural_ccn.csv'

probe = pd.read_table(FILES[2023], sep=',', header=0, nrows=5, low_memory=False)
print(f'Source reachable. {probe.shape[1]} columns per annual file.')
probe.head()

## 2.3 &nbsp; Filter to Louisiana and derive the ratio set

Each annual file covers every Medicare-certified hospital nationally. We keep
Louisiana and compute the standard financial ratios.

**One choice matters more than the rest.** Per-day ratios divide by the *actual
number of days in the reporting period*, not by 365. Cost reports do not all cover
a full year: a hospital that converts to Rural Emergency Hospital status, changes
fiscal year end, or opens mid-year files a short period. Six Louisiana reports cover
under 330 days. One covers 32 days. Dividing that hospital's cash by an annual
expense figure would report 5,140 days of cash on hand instead of 442.

In [ ]:
CONTROL = {
    1: 'Nonprofit-Church',       2: 'Nonprofit-Other',
    3: 'Proprietary-Individual', 4: 'Proprietary-Corp',
    5: 'Proprietary-Partnership', 6: 'Proprietary-Other',
    7: 'Gov-Federal',            8: 'Gov-City-County',
    9: 'Gov-County',            10: 'Gov-State',
    11: 'Gov-Hospital District', 12: 'Gov-City',
    13: 'Gov-Other',
}


def load_year(year, url):
    """Read one annual cost report file, keep Louisiana, derive the ratio set."""
    d = pd.read_table(url, sep=',', header=0, low_memory=False)
    d = d[d['State Code'] == 'LA'].copy()

    def col(name):
        if name not in d.columns:
            return pd.Series(np.nan, index=d.index)
        return pd.to_numeric(d[name], errors='coerce')

    begin  = pd.to_datetime(d['Fiscal Year Begin Date'], errors='coerce')
    end    = pd.to_datetime(d['Fiscal Year End Date'],   errors='coerce')
    period = (end - begin).dt.days.clip(lower=1)

    cash = col('Cash on Hand and in Banks').fillna(0) + col('Temporary Investments').fillna(0)
    opex = col('Less Total Operating Expense')
    dep  = col('Depreciation Cost').fillna(0)
    npr  = col('Net Patient Revenue')
    oth  = col('Total Other Income').fillna(0)

    out = pd.DataFrame({
        'fy':          year,
        'ccn':         d['Provider CCN'].astype(str).str.zfill(6),
        'hospital':    d['Hospital Name'].str.strip(),
        'city':        d['City'].str.strip().str.title(),
        'hcris_class': d['CCN Facility Type'],
        'control':     pd.to_numeric(d['Type of Control'], errors='coerce').map(CONTROL),
        'fy_days':     period,
        'npr':         npr,
    })

    # Liquidity: cash relative to daily cash operating expense, depreciation removed
    out['days_cash']     = (cash / ((opex - dep) / period)).where((opex - dep) > 0)

    # Capital structure
    out['equity_ratio']  = (col('Total Fund Balances') / col('Total Assets')
                            ).where(col('Total Assets') > 0)
    out['current_ratio'] = (col('Total Current Assets') / col('Total Current Liabilities')
                            ).where(col('Total Current Liabilities') > 0)

    # Profitability: on patient care alone, and on the bottom line
    out['pt_svc_margin'] = (col('Net Income from Service to Patients') / npr).where(npr > 0)
    out['total_margin']  = (col('Net Income') / (npr + oth)).where((npr + oth) > 0)

    print(f'  FY{year}: {len(out)} Louisiana hospitals')
    return out


print('Loading annual cost report files')
panel = pd.concat([load_year(y, u) for y, u in FILES.items()], ignore_index=True)

## 2.4 &nbsp; Restrict to the rural universe and assign ownership stratum

The 13 Type of Control codes collapse to three strata. This grouping is used
throughout as a *stratification variable*, meaning comparisons are made within it
rather than across it.

In [ ]:
xwalk = pd.read_table(CROSSWALK, sep=',', header=0, dtype=str)
xwalk['ccn'] = xwalk['ccn'].str.zfill(6)

panel = panel[panel['ccn'].isin(set(xwalk['ccn']))].copy()

panel['stratum'] = panel['control'].map(
    lambda c: 'Government'  if str(c).startswith('Gov')
    else     ('Proprietary' if str(c).startswith('Proprietary')
    else      'Nonprofit')
)

print(f"{len(panel)} hospital-years  |  {panel['ccn'].nunique()} hospitals  "
      f"|  FY{panel['fy'].min()}-FY{panel['fy'].max()}")
print()
print('Ownership stratum, most recent year')
print(panel[panel['fy'] == panel['fy'].max()]['stratum'].value_counts().to_string())

---
# 3 &nbsp; Exploratory analysis

The purpose of this section is not description for its own sake. It answers a
specific question that determines what the model may contain: **which metrics are
genuinely comparable across hospitals, and which are contaminated by structure?**

## 3.1 &nbsp; How the metrics distribute by ownership

Before testing anything, look at the shape. If liquidity separates cleanly by
ownership type, comparing a district hospital to a for-profit on days cash is
comparing two different accounting arrangements, not two levels of financial health.

In [ ]:
latest = panel[panel['fy'] == panel['fy'].max()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, metric, title in zip(
        axes,
        ['days_cash', 'pt_svc_margin'],
        ['Days cash on hand', 'Patient services margin']):

    order = ['Government', 'Nonprofit', 'Proprietary']
    data = [latest.loc[latest['stratum'] == s, metric].dropna() for s in order]
    ax.boxplot(data, tick_labels=order, showfliers=False)
    ax.set_title(title)
    ax.axhline(0, color='#c3c2b7', linewidth=0.8)

axes[0].set_ylabel('Days')
axes[1].set_ylabel('Share of net patient revenue')
plt.tight_layout()
plt.show()

print(latest.groupby('control')[['days_cash', 'equity_ratio', 'total_margin']]
      .agg(['size', 'median']).round(3)
      .sort_values(('days_cash', 'size'), ascending=False).to_string())

## 3.2 &nbsp; Confound test

Formalize what the boxplots suggest. A Kruskal-Wallis test asks whether the
distributions differ across groups, without assuming normality, which matters here
because these ratios are heavily skewed.

Two groupings are tested against two metrics. A metric that differs sharply by
ownership cannot be compared across ownership types without adjustment.

In [ ]:
print('Kruskal-Wallis tests')
print('-' * 66)

for metric in ['days_cash', 'pt_svc_margin']:
    for grouping in ['control', 'hcris_class']:
        groups = [g[metric].dropna().values
                  for _, g in latest.groupby(grouping)
                  if g[metric].notna().sum() >= 3]
        h, p = kruskal(*groups)

        if p < 0.01:
            verdict = 'CONFOUNDED'
        elif p < 0.10:
            verdict = 'borderline'
        else:
            verdict = 'clear'

        print(f'{metric:16s} ~ {grouping:12s}   H = {h:6.2f}   p = {p:.4f}   {verdict}')

### What the test establishes

**Days cash on hand is confounded by ownership** (p = 0.0001) and unrelated to
Medicare class (p = 0.22).

- Government hospital districts hold their own cash: median around **120 days**
- Proprietary corporations sweep it to a parent: median around **8 days**, often with negative equity

That gap is a treasury arrangement, not a difference in solvency.

**Patient services margin is only borderline** (p = 0.05) and clear on Medicare
class. It survives as a straight cross-hospital comparison.

**Consequence for the model:** liquidity enters as a **within-stratum percentile
rank**, so each hospital is measured against its own peer group. Operating margin
enters as-is.

In [ ]:
for col in ['days_cash', 'equity_ratio']:
    panel[f'{col}_pct'] = panel.groupby(['fy', 'stratum'])[col].rank(pct=True)

print('Raw value vs within-stratum rank')
print(panel[['hospital', 'fy', 'stratum', 'days_cash', 'days_cash_pct']]
      .head(8).round(3).to_string(index=False))

## 3.3 &nbsp; Direction of travel

A single year cannot distinguish a bad year from a decline. Plot each hospital's
liquidity across the three years, with the median overlaid.

What to look for: whether the sector median moves, and whether movement is spread
across all hospitals or concentrated in part of the distribution. Those two
patterns imply very different things about who is at risk.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))

for _, g in panel.groupby('ccn'):
    if g['days_cash'].notna().sum() >= 2:
        ax.plot(g['fy'], g['days_cash'], color='gray', alpha=0.25, linewidth=0.8)

median = panel.groupby('fy')['days_cash'].median()
ax.plot(median.index, median.values, color='#1F3864', linewidth=2.5,
        marker='o', label='Median')

ax.set_ylim(0, 400)
ax.set_xticks(sorted(panel['fy'].unique()))
ax.set_ylabel('Days cash on hand')
ax.set_title('Liquidity trajectory, one line per hospital')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

trend = panel.groupby('fy').agg(
    n=('ccn', 'size'),
    days_cash=('days_cash', 'median'),
    pt_svc_margin=('pt_svc_margin', 'median'),
    total_margin=('total_margin', 'median'),
    under_60_days=('days_cash', lambda s: int((s < 60).sum())),
    negative_margin=('total_margin', lambda s: int((s < 0).sum())),
).round(3)

print(trend.to_string())

## 3.4 &nbsp; Building hospital-level features

Reshape from one row per hospital-year to **one row per hospital**, with three
views of each metric:

- **Level** — where the hospital stands in the most recent year
- **Slope** — the direction it is moving across the three years
- **Volatility** — how much it swings, which for cost-settled hospitals is itself informative

This is what lets the clustering distinguish a hospital that is stable and weak
from one that is deteriorating.

In [ ]:
def slope(group, column):
    """Least-squares slope across available years. NaN if fewer than 2 points."""
    s = group.dropna(subset=[column])
    if len(s) < 2:
        return np.nan
    return linregress(s['fy'], s[column]).slope


METRICS = ['days_cash_pct', 'equity_ratio_pct', 'pt_svc_margin',
           'total_margin', 'days_cash', 'npr']

rows = []
for ccn, g in panel.groupby('ccn'):
    g = g.sort_values('fy')
    last = g.iloc[-1]

    row = {
        'ccn':         ccn,
        'hospital':    last['hospital'],
        'stratum':     last['stratum'],
        'control':     last['control'],
        'hcris_class': last['hcris_class'],
        'n_years':     len(g),
    }
    for m in METRICS:
        row[f'{m}_lvl']   = last[m]
        row[f'{m}_slope'] = slope(g, m)
        row[f'{m}_vol']   = g[m].std() if g[m].notna().sum() >= 2 else np.nan

    rows.append(row)

feat = pd.DataFrame(rows)
print(f'{len(feat)} hospitals, {feat.shape[1]} engineered columns')
feat.head()

## 3.5 &nbsp; Choosing which features enter the model

With 64 hospitals, a model can carry roughly **six** features before it starts
fitting noise. The candidate list is much longer than that, and many candidates
measure the same underlying thing.

Check the correlation structure, then keep one representative from each block:
**liquidity**, **capital structure**, **profitability**, **scale**.

Outliers are winsorized at the 5th and 95th percentiles rather than dropped,
because the extreme values here are real hospitals, not data errors.

In [ ]:
FEATURES = [
    'days_cash_pct_lvl',     # liquidity, ranked within ownership stratum
    'equity_ratio_pct_lvl',  # capital structure, ranked within stratum
    'pt_svc_margin_lvl',     # operating performance, where it stands
    'pt_svc_margin_slope',   # operating performance, where it is heading
    'total_margin_lvl',      # bottom line including non-patient revenue
    'npr_lvl',               # scale
]

X = feat[FEATURES].copy()
X['npr_lvl'] = np.log10(X['npr_lvl'].clip(lower=1))

complete = X.notna().all(axis=1)
Xc = X[complete].copy()

for c in Xc.columns:
    lo, hi = Xc[c].quantile([0.05, 0.95])
    Xc[c] = Xc[c].clip(lo, hi)

Xs = StandardScaler().fit_transform(Xc)
sub = feat[complete].reset_index(drop=True)

print(f'{complete.sum()} of {len(feat)} hospitals complete on all six features')
print()
print('Spearman correlation')
print(pd.DataFrame(Xs, columns=FEATURES).corr(method='spearman').round(2).to_string())

---
# 4 &nbsp; Clustering

## 4.1 &nbsp; Method, and how the result gets validated

**Ward-linkage hierarchical clustering** is used rather than k-means. It is
deterministic, it produces a dendrogram that shows where the data actually splits,
and it does not require committing to a number of groups in advance. k-means is run
alongside purely as a cross-check: if the two methods disagree about membership,
the structure is weak.

**Stability is the gate, not an afterthought.** With 64 hospitals it is trivially
easy to produce four attractive-looking groups that dissolve if two hospitals are
swapped out. The test is a bootstrap procedure:

1. Resample hospitals with replacement and cluster the resample
2. Assign **all** original hospitals to the nearest resulting cluster centre
3. Measure the Jaccard overlap between each original cluster and its best match
4. Repeat several hundred times and average

Conventional thresholds: **0.60 and above is stable**, **0.75 and above is highly
stable**, below 0.50 means the cluster is an artefact of the particular sample.

A number of groups is only reported if **every** cluster clears 0.60.

In [ ]:
def ward(X, k):
    """Ward-linkage flat clustering into k groups."""
    return fcluster(linkage(X, method='ward'), k, criterion='maxclust')


def clusterboot(X, k, n_boot=400, seed=0):
    """Bootstrap cluster stability. Returns base labels and mean Jaccard per cluster."""
    rng = np.random.default_rng(seed)
    base = ward(X, k)
    jaccard = np.zeros((n_boot, k))

    for b in range(n_boot):
        idx = rng.integers(0, len(X), len(X))
        Xb = X[idx]
        labels_b = ward(Xb, k)

        centroids = np.array([Xb[labels_b == c].mean(axis=0) for c in range(1, k + 1)])
        assigned = np.argmin(cdist(X, centroids), axis=1) + 1

        for c in range(1, k + 1):
            original = set(np.where(base == c)[0])
            best = 0.0
            for rc in range(1, k + 1):
                resampled = set(np.where(assigned == rc)[0])
                union = len(original | resampled)
                if union:
                    best = max(best, len(original & resampled) / union)
            jaccard[b, c - 1] = best

    return base, jaccard.mean(axis=0)

## 4.2 &nbsp; How many groups does the data actually support?

Three diagnostics, read together:

- **Silhouette** — how separated the clusters are. Above 0.5 is strong structure,
  0.2 to 0.3 is weak but present
- **ARI against k-means** — whether two different algorithms find the same groups
- **Bootstrap Jaccard** — whether the groups survive resampling

The number of groups is chosen on the third of these.

In [ ]:
print(f"{'k':>2}  {'silhouette':>10}  {'ARI vs kmeans':>13}   cluster sizes and stability")
print('-' * 90)

for k in range(2, 6):
    base, jac = clusterboot(Xs, k)
    km = KMeans(n_clusters=k, n_init=25, random_state=0).fit(Xs)
    sizes = np.bincount(base)[1:]

    detail = '   '.join(f'n={s:<3d} J={j:.2f}' for s, j in zip(sizes, jac))
    print(f'{k:>2}  {silhouette_score(Xs, base):>10.3f}  '
          f'{adjusted_rand_score(base, km.labels_):>13.2f}   '
          f'{detail}   mean J = {jac.mean():.3f}')

### Result

**Two groups.** Only k = 2 has every cluster above the stability threshold, at
roughly 0.65 and 0.73.

At k = 3 and above, a small cluster of four hospitals appears with a Jaccard near
0.38, meaning it is a feature of the particular sample rather than of the sector.
At k = 4 the largest cluster is the only stable one and mean stability falls to
0.53. Silhouettes sit between 0.20 and 0.24 throughout, and Ward and k-means agree
only moderately, both indicating that the underlying structure is a single split
rather than a set of tiers.

A four-tier readiness ladder would be more satisfying to present. The data does
not support one.

In [ ]:
K = 2
sub['cluster'] = ward(Xs, K)

fig, ax = plt.subplots(figsize=(13, 4.5))
dendrogram(linkage(Xs, method='ward'),
           labels=sub['hospital'].values,
           leaf_rotation=90,
           leaf_font_size=6,
           ax=ax)
ax.set_title('Ward linkage, Louisiana rural hospitals')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.show()

---
# 5 &nbsp; Interpretation

## 5.1 &nbsp; What the two groups look like

Profile each cluster on the model features plus days cash in raw units, so the
groups can be described in terms a hospital executive would recognise.

In [ ]:
profile_cols = ['days_cash_pct_lvl', 'equity_ratio_pct_lvl', 'pt_svc_margin_lvl',
                'pt_svc_margin_slope', 'total_margin_lvl', 'days_cash_lvl']

print('Cluster profile, medians')
print(sub.groupby('cluster')[profile_cols].median().round(3).to_string())
print()
print('Sizes:', np.bincount(sub['cluster'])[1:])

| | Cluster 1 (n = 27) | Cluster 2 (n = 33) |
|---|---|---|
| Patient services margin | −30% | −8% |
| Direction of travel | worsening | improving |
| Total margin | −4% | +12% |
| Days cash on hand | 39 | 108 |
| Equity percentile within stratum | 0.33 | 0.69 |

Both groups lose money delivering care, which is the normal condition for a rural
hospital in Louisiana. The difference is the size of the gap and whether it is
closing or widening.

## 5.2 &nbsp; What separates them, and what they are not

A shallow decision tree gives the actual split rule in two branches. The
cross-tabulations then check whether the clustering has simply rediscovered a
category that was already in the data.

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(Xs, sub['cluster'])
print('Split rule')
print(export_text(tree, feature_names=FEATURES))

for col in ['hcris_class', 'stratum']:
    print(f'Cluster by {col}')
    print(pd.crosstab(sub['cluster'], sub[col]).to_string())
    print()

The split runs on total margin and equity rank, and it is spread across both
Medicare class and ownership stratum. It is a genuinely new grouping rather than a
relabelling of CAH status or of who owns the hospital.

## 5.3 &nbsp; Do the existing program groupings track financial position?

A different question, and the one with the most direct programmatic consequence.
Take each grouping we already have, and try to predict membership from the
financial features alone.

If accuracy sits at or below the base rate, that grouping carries no information
about financial position, and it cannot be used as a stand-in for readiness.

In [ ]:
flags = xwalk[['ccn', 'lihnc_member', 'in_epic_pipeline']]
sub = sub.merge(flags, on='ccn', how='left')

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)

targets = {
    'CAH designation': (sub['hcris_class'] == 'CAH').astype(int),
    'LIHNC member':    (sub['lihnc_member'] == 'Yes').astype(int),
    'Epic pipeline':   (sub['in_epic_pipeline'] == 'Yes').astype(int),
}

print(f"{'Grouping':<18}{'base rate':>11}{'CV accuracy':>13}{'lift':>8}   verdict")
print('-' * 72)

for name, y in targets.items():
    base_rate = max(y.mean(), 1 - y.mean())
    acc = cross_val_score(LogisticRegression(C=0.5, max_iter=2000),
                          Xs, y, cv=cv, scoring='accuracy').mean()
    lift = acc - base_rate
    verdict = 'carries signal' if lift > 0.05 else 'no financial signal'
    print(f'{name:<18}{base_rate:>11.2f}{acc:>13.2f}{lift:>+8.2f}   {verdict}')

### Reading the result

**CAH designation is predictable** from financial position. Reimbursement class and
finances are linked, which is what the reasoning in section 1 anticipated.

**LIHNC membership is not.** Neither is **Epic pipeline participation**. Both score
*below* the base rate, meaning the financial features are worse than simply guessing
the majority.

Whatever determined who joined the network and who entered the Epic waves,
financial position was not part of it. Two consequences follow. Neither grouping can
serve as a proxy for readiness. And if either program is intended to reach hospitals
under financial pressure, it is not currently selecting for that.

## 5.4 &nbsp; Scoring VBC potential

Section 1 defined VBC potential as requiring three things. Two can be measured from
cost report data. The third cannot be measured from any available dataset, so it is
carried as an explicit blank rather than estimated.

| Axis | Source | Status |
|---|---|---|
| **Risk-bearing capacity** | Liquidity and equity, ranked within ownership stratum | Scored |
| **Operating performance** | Patient services margin, its trend, and total margin | Scored |
| **Attribution capacity** | Primary care providers billing under the hospital's own TIN, and certified provider-based RHCs | **No data. Requires a survey** |

The axes are reported separately rather than combined into one number. A hospital
strong on both scored axes and empty on the third is still not a candidate, and
collapsing them into a single score would hide exactly that case.

In [ ]:
z = pd.DataFrame(Xs, columns=FEATURES)

sub['risk_capacity']  = z[['days_cash_pct_lvl', 'equity_ratio_pct_lvl']].mean(axis=1)
sub['operating_perf'] = z[['pt_svc_margin_lvl', 'pt_svc_margin_slope',
                           'total_margin_lvl']].mean(axis=1)
sub['attribution']    = np.nan   # awaiting the primary care and billing TIN survey

for c in ['risk_capacity', 'operating_perf']:
    sub[f'{c}_pct'] = sub[c].rank(pct=True).round(2)

fig, ax = plt.subplots(figsize=(7.5, 6))

palette = {1: ('#E24B4A', 'Cluster 1  (n=27)'),
           2: ('#1D9E75', 'Cluster 2  (n=33)')}

for cl, (colour, label) in palette.items():
    s = sub[sub['cluster'] == cl]
    ax.scatter(s['operating_perf'], s['risk_capacity'], c=colour, s=48,
               alpha=0.75, edgecolor='white', linewidth=0.5, label=label)

ax.axhline(0, color='#c3c2b7', linewidth=0.8)
ax.axvline(0, color='#c3c2b7', linewidth=0.8)
ax.set_xlabel('Operating performance  (standardized)')
ax.set_ylabel('Risk-bearing capacity  (within ownership stratum)')
ax.set_title('Two scored axes. The third, attribution, has no data.')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
scorecard = (sub[['hospital', 'hcris_class', 'stratum', 'cluster',
                  'risk_capacity_pct', 'operating_perf_pct', 'attribution',
                  'days_cash_lvl', 'pt_svc_margin_lvl', 'total_margin_lvl']]
             .sort_values('operating_perf_pct', ascending=False)
             .round(3))

scorecard.to_csv('la_hospital_scorecard.csv', index=False)
print(f'Wrote la_hospital_scorecard.csv  ({len(scorecard)} hospitals)')

# In Colab: from google.colab import files; files.download('la_hospital_scorecard.csv')

scorecard.head(15)

## 5.5 &nbsp; What this analysis does not answer

These belong with any version of the output.

1. **Cluster membership is provisional.** Stability of 0.65 and 0.73 supports
   reporting a two-group split at the sector level. It does not support ranking or
   labelling a named hospital, and a single hospital's position should be checked
   against its own three-year record before it is used in any conversation.

2. **There is no attribution measure.** The count of primary care providers billing
   under each hospital's own TIN, and of certified provider-based rural health
   clinics, is in no public dataset. It requires a short survey of the membership,
   and it is the single most decisive missing input for the VBC question.

3. **There is no quality measure.** Financial capacity is one of two criteria for
   selecting VBC participants and only one is present here. Care Compare and MIPS
   results need to be added before any candidate list is published.

4. **Non-patient revenue is not decomposed.** Total Other Income appears as a single
   line, so ad valorem tax, cost settlement, grants and investment income cannot be
   separated. For a parish hospital district, much of the gap between patient
   services margin and total margin may be property tax rather than Medicaid policy,
   and those two have very different outlooks.

5. **Nothing here is causal.** 64 hospitals over three years with no exogenous
   variation supports description, not inference about what would happen under a
   different policy.

6. **Coverage.** Four hospitals are excluded for incomplete features, and two have
   fewer than three years of filings.